In [147]:
import numpy as np
import pandas as pd
import torch
from core.trainers.trainer_v2 import SparseUnionAxisBoxes, train_model
from sample_predicates_data import sample_binary_predicates, sample_numeric_predicates, sample_numeric_data, sample_binary_data

In [139]:
num_predicates_avg = 1
num_predicates_max = 1

num_numeric_avg = 1
num_numeric_max = 1
num_numeric_bins_avg = 1
num_numeric_bins_max = 15
size_numeric_bins_max = 5

num_binary_avg = 2
num_binary_max = 5

num_rows = 1000

In [140]:
numeric_predicates, numeric_clauses = sample_numeric_predicates(
    num_predicates_avg,
    num_predicates_max,
    num_numeric_avg,
    num_numeric_max,
    num_numeric_bins_avg,
    num_numeric_bins_max,
    size_numeric_bins_max
)
binary_predicates, binary_clauses = sample_binary_predicates(
    num_predicates_avg,
    num_predicates_max,
    num_binary_avg,
    num_binary_max
)

In [141]:
numeric_clauses['predicate_0']

{'numeric_0': [(7, 11)]}

In [142]:
binary_clauses['predicate_0']

['binary_0', 'binary_1', 'binary_3']

In [143]:
numeric_data_in = sample_numeric_data(num_numeric_avg, num_numeric_max, num_rows//2, clauses=numeric_clauses['predicate_0'])
binary_data_in = sample_binary_data(num_binary_avg, num_binary_max, num_rows//2, predicate=binary_predicates['predicate_0'])

In [144]:
numeric_data_out = pd.DataFrame(np.random.choice(range(num_numeric_bins_max), size=numeric_data_in.shape), columns=numeric_data_in.columns)
binary_data_out = pd.DataFrame(np.random.binomial(1, p=.5, size=binary_data_in.shape), columns=binary_data_in.columns)

In [145]:
numeric_data_in = numeric_data_in/num_numeric_bins_max
numeric_data_out = numeric_data_out/num_numeric_bins_max
numeric_data = pd.concat([numeric_data_in, numeric_data_out]).reset_index(drop=True)
binary_data = pd.concat([binary_data_in.assign(label=1), binary_data_out.assign(label=0)]).reset_index(drop=True)
data = pd.concat([numeric_data, binary_data], axis=1)

In [134]:
model = SparseUnionAxisBoxes(1, n_boxes=6, feature_lambda=1e-3, box_lambda=1e-2, k=10.0)

In [136]:
trained_model = train_model(
    model,
    torch.from_numpy(data['numeric_0'].astype(float).values[:,None]),
    torch.from_numpy(data['label'].astype(float).values),
    lr=1e-2,
    epochs=1000,
    use_balanced_loss=False,
    box_lambda_warmup_epochs=250
)

epoch    0 | loss 1.7622 | box_λ_scale 0.00 | active boxes 6
epoch  100 | loss 0.6931 | box_λ_scale 0.40 | active boxes 6
epoch  200 | loss 0.6995 | box_λ_scale 0.80 | active boxes 6
epoch  300 | loss 0.6995 | box_λ_scale 1.00 | active boxes 4
epoch  400 | loss 0.6969 | box_λ_scale 1.00 | active boxes 4
epoch  500 | loss 0.6955 | box_λ_scale 1.00 | active boxes 3
epoch  600 | loss 0.6945 | box_λ_scale 1.00 | active boxes 3
epoch  700 | loss 0.6936 | box_λ_scale 1.00 | active boxes 3
epoch  800 | loss 0.6928 | box_λ_scale 1.00 | active boxes 3
epoch  900 | loss 0.6921 | box_λ_scale 1.00 | active boxes 3
